# ตรวจความชื้นดินช่วงฟื้นต้น — 19 กันยายน 2026

## TL;DR
สองเหตุการณ์ที่มีสัญญาณฝนใกล้จุดสูงสุด วันที่ 12 และ 13 กันยายน มีค่าดินลดจาก 62.7 และ 58.1% เหลือ 43.7 และ 43.8% หลัง 3 ชั่วโมง ไม่ใช่หลักฐานว่าทุเรียนมีน้ำพอหรือขาดน้ำ

Notebook นี้เป็นการตรวจเชิงตัวเลข ไม่ใช่โมเดลพยากรณ์หรือคำสั่งให้น้ำ และไม่เขียนกลับ Google Sheets

## Context and methods
ผู้ใช้แจ้งว่าทุเรียนหมอนทองอายุ 9 ปีอยู่ช่วงฟื้นต้น เซนเซอร์แนวตั้งชายพุ่ม ลงจากผิวดิน 12 นิ้ว ยังไม่มีข้อมูลสอบเทียบหรือช่วงรากที่ดูดน้ำจริง

[แหล่งต้นทาง: Farm IoT Data](https://docs.google.com/spreadsheets/d/1-CDRERyx20UNCy5Oq3a8wt4lv3SmBx9nH2Kf2IQZvVE/edit), แท็บ `readings_15min`, A1:Z2410; สำเนา 2026-09-19 06:24:05 UTC; เวลาทั้งหมดที่รายงานเป็น Asia/Bangkok

เลือกกลุ่มค่าดิน >=50% และจุดสูงสุดของกลุ่ม โดยแยกเมื่อข้อมูลห่าง >30 นาที ตรวจสัญญาณฝน ±1 ชั่วโมง หาจุด +1/+2/+3 ชั่วโมงแบบเวลาตรงกัน ไม่ประมาณค่าขาด เกณฑ์ 50/40 ใช้ตรวจข้อสังเกต ไม่ใช่เกณฑ์พืชขาดน้ำ


In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'tools/analyze_live_soil_drydown.py').exists())
if str(ROOT / 'tools') not in sys.path:
    sys.path.insert(0, str(ROOT / 'tools'))
from analyze_live_soil_drydown import inspect_snapshot
df, profile, daily, events, gaps, flatlines = inspect_snapshot()
print(json.dumps(profile, ensure_ascii=False, indent=2))

{
  "rows": 2409,
  "columns_source": 26,
  "first_th": "2026-08-14T10:45:00+07:00",
  "last_th": "2026-09-19T13:15:00+07:00",
  "expected_15min_slots": 3467,
  "missing_15min_slots": 1058,
  "off_grid_rows": 0,
  "gaps_over_30min": 40,
  "largest_gap_hours": 11.0,
  "original_weather_signature_missing": 44,
  "additional_weather_flatline_rows": 1365,
  "additional_weather_flatline_runs": 44,
  "weather_signature_usable_rows": 1000,
  "soil_min": 22.1,
  "soil_max": 85.8,
  "soil_latest": 31.3,
  "timestamp_th_max_error_seconds": 2.14e-07,
  "snapshot_sha256": "1dd33b80c50222f88366d4c73a87a9d43c24231ba8b2ab2a78d0467a8dc0505d",
  "source": {
    "spreadsheet_id": "1-CDRERyx20UNCy5Oq3a8wt4lv3SmBx9nH2Kf2IQZvVE",
    "sheet": "readings_15min",
    "sheet_id": 1874414600,
    "ranges": [
      "readings_15min!A1:Z500",
      "readings_15min!A501:Z1000",
      "readings_15min!A1001:Z1500",
      "readings_15min!A1501:Z2000",
      "readings_15min!A2001:Z2410"
    ],
    "time_zone": "Asia/Ba

## Data quality
กฎอากาศค้างคือ outdoor temperature, outdoor humidity, pressure, light และ UV ไม่เปลี่ยนพร้อมกันอย่างน้อย 60 นาที โดยแยกเมื่อช่วงห่าง >30 นาที ปิดใช้คอลัมน์อากาศเฉพาะสำเนาวิเคราะห์ แต่คงข้อมูลดินทุกแถว กฎนี้อาจจับช่วงนิ่งจริงด้วย ไม่ใช่เวลาปลั๊กหลุดที่ยืนยันแล้ว

ความครบถ้วนคำนวณจากสมมติฐานว่าส่งทุก 15 นาทีตลอดช่วงแรกถึงสุดท้าย ไม่มีการสรุปสาเหตุข้อมูลขาด และไม่บวกค่าฝน rolling 1h/24h เป็นผลรวมฝนรายวัน


In [2]:
print(f"Observed slots: {profile['rows']}/{profile['expected_15min_slots']}")
print(f"Missing slots: {profile['missing_15min_slots']}")
print(f"Suspected weather flatlines: {profile['additional_weather_flatline_rows']} rows")
print(daily.tail(12).to_string(index=False))
print(gaps.tail(10).to_string(index=False))

Observed slots: 2409/3467
Missing slots: 1058
Suspected weather flatlines: 1365 rows
   date_th  rows  soil_min  soil_max  soil_first  soil_last  weather_flatline_rows  rain_1h_available_rows  rain_1h_positive_rows
2026-09-08    81      24.0      24.4        24.4       24.0                     81                       0                      0
2026-09-09    64      23.7      24.6        24.0       24.4                     56                       8                      0
2026-09-10    64      23.9      42.1        24.1       34.8                      0                      61                     15
2026-09-11    60      32.1      41.7        41.7       37.5                      0                      60                     10
2026-09-12    59      32.1      62.7        33.9       40.0                      0                      57                     21
2026-09-13    55      31.0      58.1        35.4       42.6                      0                      54                     16
2026-

## Results: first three hours
แสดงทุกเหตุการณ์ที่ถึง 50% ก่อน จึงไม่ซ่อนกรณีที่ข้อมูลฝนใช้ไม่ได้ `rain_not_established` ไม่ได้แปลว่าไม่มีฝน


In [3]:
cols = ['peak_time_th', 'peak_sheet_row', 'peak_moisture',
        'moisture_1h', 'moisture_2h', 'moisture_3h', 'rain_attribution']
print(events[cols].to_string(index=False))

confirmed = events.loc[events.rain_attribution.eq('rain_signal_present')].copy()
assert confirmed.peak_sheet_row.tolist() == [1986, 2055]
assert confirmed.peak_moisture.tolist() == [62.7, 58.1]
assert confirmed.moisture_3h.tolist() == [43.7, 43.8]
print('Both selected 0-3h windows cross-checked against live formatted Sheet ranges.')

             peak_time_th  peak_sheet_row  peak_moisture  moisture_1h  moisture_2h  moisture_3h     rain_attribution
2026-08-26T13:00:00+07:00             774           85.8          NaN          NaN         45.8 rain_not_established
2026-08-30T13:00:00+07:00            1076           68.6         52.8         46.0         43.6 rain_not_established
2026-09-02T17:00:00+07:00            1327           61.4         48.6         44.2         42.1 rain_not_established
2026-09-03T17:15:00+07:00            1389           84.1         59.0         46.7         45.2 rain_not_established
2026-09-12T14:00:00+07:00            1986           62.7         47.0         45.1         43.7  rain_signal_present
2026-09-13T17:00:00+07:00            2055           58.1         48.8         45.4         43.8  rain_signal_present
2026-09-17T12:15:00+07:00            2266           54.5         47.2         45.5         44.2 rain_not_established
Both selected 0-3h windows cross-checked against live formatted 

In [4]:
# Display the 26 exact observations, not interpolated values.
for peak in confirmed.itertuples():
    t = pd.Timestamp(peak.peak_time_th)
    window = df.loc[df.timestamp_th.between(t, t + pd.Timedelta(hours=3))]
    assert len(window) == 13
    assert window.timestamp.diff().iloc[1:].eq(pd.Timedelta(minutes=15)).all()
    print(window[['sheet_row', 'timestamp_th', 'soil_moisture_percent']].to_string(index=False))

 sheet_row              timestamp_th  soil_moisture_percent
      1986 2026-09-12 14:00:00+07:00                   62.7
      1987 2026-09-12 14:15:00+07:00                   53.8
      1988 2026-09-12 14:30:00+07:00                   50.3
      1989 2026-09-12 14:45:00+07:00                   48.4
      1990 2026-09-12 15:00:00+07:00                   47.0
      1991 2026-09-12 15:15:00+07:00                   46.2
      1992 2026-09-12 15:30:00+07:00                   45.8
      1993 2026-09-12 15:45:00+07:00                   45.4
      1994 2026-09-12 16:00:00+07:00                   45.1
      1995 2026-09-12 16:15:00+07:00                   44.7
      1996 2026-09-12 16:30:00+07:00                   44.2
      1997 2026-09-12 16:45:00+07:00                   44.0
      1998 2026-09-12 17:00:00+07:00                   43.7
 sheet_row              timestamp_th  soil_moisture_percent
      2055 2026-09-13 17:00:00+07:00                   58.1
      2056 2026-09-13 17:15:00+07:00    

## Takeaways and limits

- การลดเร็วช่วงแรกเกิดขึ้นจริง แต่ไม่ทราบสัดส่วนที่เป็นการระบายน้ำ การกระจายตัว น้ำที่พืชใช้ หรือความคลาดเคลื่อนหัววัด
- วันที่ 12 กันยายนพบค่าครั้งแรก <=40% เวลา 21:15; วันถัดไปเวลา 14:00 อ่านได้ 33.1% นี่คือค่าที่วัดตามเวลา ไม่ใช่การยืนยันว่าตลอดช่วงไม่มีฝนซ้ำ
- วันที่ 13 กันยายนมีช่องว่างข้อมูลก่อนพบค่า <=40% จึงไม่ใช้ค่านั้นระบุเวลาข้ามเกณฑ์ที่แน่นอน
- ค่าฝนใช้ไม่ได้ในหลายช่วง ไม่ใช้ค่าว่างแทนศูนย์และไม่เรียกวันนั้นว่าไม่มีฝน
- จุด +24/+48 ชั่วโมงอาจมีฝนซ้ำ ยังไม่ได้ตัดช่วงที่เปียกซ้ำออกจาก dry-down event
- เซนเซอร์หนึ่งจุดไม่เป็นตัวแทนน้ำทั้งเขตราก ค่า 31.3% ล่าสุดไม่ได้แปลว่าน้ำเหลือให้ใช้ 31.3%
- ขั้นต่อไป: เทียบกับดินจริง/การสอบเทียบ และวัดการเก็บน้ำของดินก่อนกำหนดเกณฑ์แจ้งเตือนในช่วงฟื้นต้น ไม่มีเกณฑ์ปั๊มหรือปริมาณลิตรที่ยืนยันจากการตรวจครั้งนี้

## Sources and rerun
[University of Minnesota Extension — Soil moisture sensors for irrigation scheduling](https://extension.umn.edu/natural-resources/conservation/agricultural-soil-and-water/irrigation/soil-moisture-sensors-for-irrigation-scheduling) อธิบาย field capacity, available water และการสอบเทียบ เป็นหลักการทั่วไป ไม่ใช่ค่ามาตรฐานทุเรียน

โค้ด `tools/analyze_live_soil_drydown.py`; สำเนา `data/analysis_soil_2026-09-19/live_readings_snapshot.json`; รายละเอียด `docs/SOIL_DRYDOWN_REVIEW_2026-09-19_TH.md` ภายในโครงการนี้

เปิดใน VS Code เลือก Python `.build/national-analysis-venv` แล้ว Run All ไม่ต้องล็อกอิน GitHub สำเนาดิบอยู่ในเครื่องเท่านั้น ไม่ถูก commit หรือ push
